In [ ]:
from Hand import Hand
from Controller import Controller
from filter.KalmanFilter import KalmanPosVelFilter
from filter.OneEuroFilter import OneEuroFilter

import math
import time
from enum import Enum, auto
import platform

# utils
from lib.utils import *
from lib.math_utils import *
from lib.draw_utils import *

import cv2
import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(precision=4)


# hand landmark: https://google.github.io/mediapipe/solutions/hands.html
import mediapipe as mp 
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

In [ ]:
def preprocess_img(image):
  h, w = image.shape[:2]
  # resize
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h/(w/DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w/(h/DESIRED_HEIGHT)), DESIRED_HEIGHT))

  return img


In [ ]:
class FINGER_STATE(Enum):
  BENT = 1
  STRAIGHT = 0
  UNKNOWN = auto()

class FINGER(Enum):
  THUMB = 0
  INDEX = 1
  MIDDLE = 2
  RING = 3
  PINKY = 4


def get_finger_state(landmarks):
  assert landmarks.shape == (21, 3)
  # TODO: detect thumb state
  res = np.array([FINGER_STATE.UNKNOWN for i in range(5)])
  
  # finger_idx
  # thumb: 1~4, index: 5~8, middle: 9~12, ring: 13~16, pinky: 17~20
  for finger_idx in range(1, 21, 4):
    tip = landmarks[finger_idx+3, :]
    dip = landmarks[finger_idx+2, :]
    pip = landmarks[finger_idx+1, :]
    mcp = landmarks[finger_idx, :]

    accumulated_angle = (np.pi - angle_between_vectors(pip - mcp, pip - dip)) + (np.pi - angle_between_vectors(dip - pip, dip - tip))
    
    if finger_idx == 0:
      # NOTE: special case -> thumb
      res[finger_idx] = FINGER_STATE.BENT if accumulated_angle > (np.pi * 0.25) else FINGER_STATE.STRAIGHT
    else :
      res[(finger_idx-1) // 4] = FINGER_STATE.BENT if accumulated_angle > (np.pi * 0.4) else FINGER_STATE.STRAIGHT

  return res

In [ ]:
def get_existence_filter(should_save=False):
  P = 1.
  R = .25
  Q = .5
  # return KalmanPosVelFilter(dim_z=1, P=P, R=R, Q=Q, should_save=should_save)
  return OneEuroFilter()

def get_handedness_filter(should_save=False):
  P = 1.
  R = .25
  Q = .5
  # return KalmanPosVelFilter(dim_z=1, P=P, R=R, Q=Q, should_save=should_save)
  return OneEuroFilter()

In [ ]:
def get_lm_kalman_filter(image_width, image_hight, image_depth, match_bone_length=False, should_save=False):
    # landmarks P
    pos_P = (image_width/2, image_hight /
                2, image_depth/2)
    vel_P = (image_width/4, image_hight /
                4, image_depth/2)
    P = np.eye( 126) * np.tile(np.stack([pos_P, vel_P], axis=1), (21, 1, 1)).flatten()
    # landmarks Q
    pos_Q = (4., 4., 4.)
    vel_Q = (12., 7., 7)
    Q_corr = np.array([
                [0.3511, 0.349, 0.0004],
                [0.5964, 0.3559, 0.2361],
                [0.7705, 0.3104, 0.4081],
                [0.8716, 0.3043, 0.5259],
                [1., 0.3614, 0.6697],
                [0.5417, 0.2475, 0.5229],
                [0.6963, 0.2957, 0.7271],
                [0.8096, 0.5599, 0.8767],
                [0.9454, 0.964, 1.],
                [0.4744, 0.2455, 0.4395],
                [0.6485, 0.3047, 0.7169],
                [0.6545, 0.591, 0.8171],
                [0.6612, 1., 0.8971],
                [0.4781, 0.2569, 0.358],
                [0.6344, 0.3066, 0.5761],
                [0.6231, 0.465, 0.6243],
                [0.6208, 0.7028, 0.6646],
                [0.487, 0.2692, 0.3178],
                [0.6103, 0.2947, 0.456],
                [0.6283, 0.3491, 0.5105],
                [0.6442, 0.4335, 0.5658],
            ])
    lm_pos_Q = np.tile(pos_Q, [21, 1])
    lm_vel_Q = vel_Q * Q_corr
    Q = np.eye( 126) * np.stack([lm_pos_Q, lm_vel_Q], axis=2).flatten()
    # landmarks R
    if match_bone_length:
        pos_R = (5., 5., 10.)
    else:
        pos_R = (10, 7, 15.)
    R_block = np.eye(3) * pos_R
    R = block_diagonal_array(63//3, R_block)

    return KalmanPosVelFilter(dim_z=21*3, P=P, R=R, Q=Q, should_save=should_save)

def get_lm_oneeuro_filter(should_save=False):
    return OneEuroFilter(min_cutoff=.2, beta=.03, d_cutoff=1., should_save=should_save)

In [ ]:
DEBUG_MOUSE = False
DEBUG_KEYBOARD = False 

DESIRED_HEIGHT = 720 
DESIRED_WIDTH = 720 

# DEV:
MATCH_REAL_BONES_LENGTH = False 

In [ ]:
controller = Controller()
controller.print_system_info()

if DEBUG_MOUSE:
  mouse_listener = controller.get_mouse_listener()
  mouse_listener.start()
  mouse_listener.wait()
if DEBUG_KEYBOARD:
  keyboard_listener = controller.get_keyboard_listener()
  keyboard_listener.start()
  keyboard_listener.wait()

# fps
s_time = time.time()
frame_cnt = 0
prev_frame_cnt = 0
prev_timestamp = time.time()

# filter
ex_filter = get_existence_filter()
hn_filter = get_handedness_filter()
lm_filter = get_lm_oneeuro_filter()

# gesture
click_queue = np.ones(4)
cursor_filter = OneEuroFilter(0.1, 1.3e-3, d_cutoff=1, should_save=False)
cursor_filter.build(z=[0, 0])
scroll_filter = OneEuroFilter(125, 30, d_cutoff=1, should_save=True)
scroll_filter.build(z=[0, 0])
SCROLL_FACTOR = 0.02

# utility
is_first = True
depth_factor = 1
prev_t = time.time()

# DEV:
prev_landmarks_x = None

cap = cv2.VideoCapture(0)
with mp_hands.Hands(
    min_detection_confidence=0.75,
    min_tracking_confidence=0.7) as hands:
  print(cap.isOpened())
  while cap.isOpened():
    success, raw_image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # Flip the image horizontally for a later selfie-view display, and convert the BGR image to RGB.
    image = cv2.cvtColor(cv2.flip(preprocess_img(raw_image), 1), cv2.COLOR_BGR2RGB)
    # To improve performance, optionally mark the image as not writeable to pass by reference.
    image.flags.writeable = False
    results = hands.process(image)
    
    image_hight, image_width, _ = image.shape
    # Draw the hand annotations on the image.
    image.flags.writeable = True
    annotated_image = image


    # measurement
    raw_handedness = None
    raw_landmarks = None
    z_existence = 0
    z_handedness = None
    z_landmarks = None
    if bool(results.multi_hand_landmarks):
      raw_handedness = results.multi_handedness[0].classification[0].index
      raw_landmarks = results.multi_hand_landmarks[0]
      z_existence = 1
      z_handedness = .5 + (.5 if raw_handedness == 1 else -.5) * results.multi_handedness[0].classification[0].score 
      z_landmarks = np.empty((21, 3))
      for landmark_idx in mp_hands.HandLandmark:
          z_landmarks[landmark_idx] = np.array([
              raw_landmarks.landmark[landmark_idx].x,
              raw_landmarks.landmark[landmark_idx].y,
              raw_landmarks.landmark[landmark_idx].z,
          ])
      z_landmarks = z_landmarks * (image_width, image_hight, image_width)
      
    # evolve estimate
    if is_first == True:
      hand = Hand.Hand(
        ex_filter,
        hn_filter,
        lm_filter,
        MATCH_REAL_BONES_LENGTH
      )
      hand.build(z_existence, z_handedness, z_landmarks)
      prev_landmarks_x = hand.landmarks_x
      is_first = False
    else:
      prev_landmarks_x = hand.landmarks_x
      hand.update(z_existence, z_handedness, z_landmarks)

    # save
    # WARN: only save when getting z. Otherwise, `Saver.z` and `Saver.x_post` would contain `None` which cause error when reshaping array
    if z_existence == 1:
      hand.save()


    # set alias
    existence_x = hand.existence_x
    handedness_x = hand.handedness_x
    landmarks_x = hand.landmarks_x
    existence_dx = hand.existence_dx
    handedness_dx = hand.handedness_dx
    landmarks_dx = hand.landmarks_dx
    depth_factor = hand.depth_factor if hand.depth_factor else depth_factor
    dt = time.time() - prev_t
    prev_t = dt + prev_t


    if existence_x > .5:
      # Gesture
      finger_states = get_finger_state(landmarks_x)
      te = np.linalg.norm(landmarks_x[4, :] - landmarks_x[6, :]) < 50.
      ie = np.linalg.norm(landmarks_x[8, :] - landmarks_x[0, :]) > 100.
      me = np.linalg.norm(landmarks_x[12, :] - landmarks_x[0, :]) > 120.
      re = np.linalg.norm(landmarks_x[16, :] - landmarks_x[0, :]) > 120.
      pe = np.linalg.norm(landmarks_x[20, :] - landmarks_x[0, :]) > 120.
      
      """
      # Mouse click / drag
      ## prev
      # NOTE: `ti` cannot separate idle and click clearly 
      # if ti:
      if finger_states[0] == FINGER_STATE.STRAIGHT:
        if np.all(click_queue == FINGER_STATE.BENT.value):
          controller.mouse_release('left')
          draw_click_drag(annotated_image, landmarks_x, is_click=False, is_drag=False)

        elif np.all(click_queue[:-1] == FINGER_STATE.BENT.value) or np.all(click_queue[:-2] == FINGER_STATE.BENT.value) or np.all(click_queue[:-3] == FINGER_STATE.BENT.value):
          # FUTURE: current condition cannot support "double click"
          controller.mouse_click('left')
          draw_click_drag(annotated_image, landmarks_x, is_click=True, is_drag=False)
      # NOTE: `ti` cannot separate idle and click clearly
      # elif not ti: 
      elif finger_states[0] == FINGER_STATE.BENT:
        if np.all(click_queue[:-1] == FINGER_STATE.BENT.value):
          if click_queue[-1] == FINGER_STATE.STRAIGHT.value:
            controller.mouse_press('left')
          draw_click_drag(annotated_image, landmarks_x, is_click=False, is_drag=True)
      ## update
      if finger_states[0].value != FINGER_STATE.UNKNOWN:
        click_queue = np.roll(click_queue, 1)
        # NOTE: `ti` cannot separate idle and click clearly
        # click_queue[0] = FINGER_STATE.BENT.value if ti else FINGER_STATE.STRAIGHT.value
        click_queue[0] = finger_states[0].value
      """

      # Move mouse
      if ie and not me:
      # if np.all(finger_states[[2,3]] == FINGER_STATE.BENT) and (
            # (np.all(np.abs(landmarks[0, 0:2, 1]) < 10.))
            # or 
            # (finger_states[1] == FINGER_STATE.STRAIGHT)
          # ):
        dx = (landmarks_x[8, 0] - prev_landmarks_x[8, 0]) / dt
        dy = (landmarks_x[8, 1] - prev_landmarks_x[8, 1]) / dt
        cursor_pos = cursor_filter.update([dx, dy])
        cursor_filter.save()
        controller.mouse_move(cursor_pos[0], cursor_pos[1])
        # controller.mouse_move(dx, dy)
        # cv2.putText(annotated_image, f'{cursor_pos[0]:.0f}, {cursor_pos[1]:.0f}',
        #             org=(image_width//2-100, 30),  # bottomLeftCornerOfText
        #             fontFace=cv2.FONT_HERSHEY_SIMPLEX,
        #             fontScale=1,
        #             color=COLOR.red,
        #             lineType=2)
      else:
        cursor_pos = cursor_filter.update([0, 0])
        cursor_filter.save()

      # DEV:
      # dx = landmarks_dx[8, 0] 
      # dy = landmarks_dx[8, 1] 
      # dx = (landmarks_x[8, 0] - prev_landmarks_x[8, 0]) / dt
      # dy = (landmarks_x[8, 1] - prev_landmarks_x[8, 1]) / dt
      # controller.mouse_move(dx, dy)

      """
      # Scroll vertically
      dy = landmarks_dx[8, 1]
      scroll_vel = scroll_filter.update([0, dy])*SCROLL_FACTOR

      ## Scroll down
      if ie and me and not re:
      # if np.all(finger_states[[3,4]] == FINGER_STATE.BENT) and \
      #       np.all(finger_states[[1,2]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(scroll_vel[1], -controller.MAX_SCROLL_SPEED, 0)
        controller.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks_x[8, 0:2].astype(int), (landmarks_x[8, 0:2] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)
        cv2.putText(annotated_image, f'{scroll_y:.3f}',
                    org=(image_width//2+100, 30),  # bottomLeftCornerOfText
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1,
                    color=COLOR.red,
                    lineType=2)

      ## Scroll up 
      if ie and me and re and not pe:
      # if np.all(finger_states[[4]] == FINGER_STATE.BENT) and \
      #       np.all(finger_states[[1,2,3]] == FINGER_STATE.STRAIGHT):
        scroll_y = np.clip(-scroll_vel[1], 0, controller.MAX_SCROLL_SPEED)
        controller.scroll(0, scroll_y)
        # annotate
        cv2.arrowedLine(annotated_image, landmarks_x[8, 0:2].astype(int), (landmarks_x[8, 0:2] + [0, -scroll_y*10]).astype(int), color=(255, 50, 50), thickness=3)
        cv2.putText(annotated_image, f'{scroll_y:.3f}',
                    org=(image_width//2+100, 30),  # bottomLeftCornerOfText
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1,
                    color=COLOR.red,
                    lineType=2)
      """


      """
      # Switch Desktop (4 l/r)
      if np.all(finger_states[[1,2,3,4]] == FINGER_STATE.STRAIGHT) and \
        finger_states[0] == FINGER_STATE.BENT:
        if landmarks_dx[8, 0] > 150:
          controller.switch_desktop_left()
        elif landmarks_dx[8, 0] < -150:
          controller.switch_desktop_right()
        elif landmarks_dx[8, 1] < -150:
          controller.show_control_center()
        elif landmarks_dx[8, 1] > 150:
          controller.show_app_expose()
      """

      # LaunchPad (3 in)
      # Show Desktop (3 out)

      # Draw 
      # measurement landmarks
      if z_existence:
        mp_drawing.draw_landmarks(annotated_image, results.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS)
        # draw_landmarks(annotated_image, z_landmarks)
        # left_side_landmarks = z_landmarks[:, [2, 1, 0]] * (-1, 1, 0)
        # draw_landmarks(annotated_image, left_side_landmarks, origin=(image_width*.2, image_hight*.5, 0))

      # Draw normal landmark
      draw_landmarks(annotated_image, landmarks_x, origin=np.asarray((landmarks_x[0, 0], landmarks_x[0, 1], 0)))

      # Draw landmarks seeing from x-axis
      left_side_landmarks = landmarks_x[:, [2, 1, 0]] * (-1, 1, 0)
      draw_landmarks(annotated_image, left_side_landmarks, origin=(image_width*.2, image_hight*.5, 0))

      # Draw projected landmarks
      # transformed_landmarks, rotation_matrix, delta_origin = Hand.Hand.transform_to_palm_coordinate(landmarks_x[:, :], handedness_x[0] > .5)
      # draw_landmarks(annotated_image, transformed_landmarks, origin=(image_width*.2, image_hight*.5, 0))
      # inv_landmarks = Hand.Hand.reverse_landmarks_transformation(transformed_landmarks, rotation_matrix, delta_origin) 
      # draw_landmarks(annotated_image, inv_landmarks)
      # Draw projection axis
      # draw_3axis(annotated_image, rotation_matrix, delta_origin)      

      # Draw handedness
      # draw_handedness(annotated_image, handedness_x)

      # Draw finger states
      # draw_finger_state(annotated_image, handedness_x, finger_states)
      
    else: 
      # TODO: reduce to default position and uncertainty
      pass

    frame_cnt += 1
    now_timestamp = time.time()
    if now_timestamp - prev_timestamp >= 1:
      prev_timestamp, prev_frame_cnt = now_timestamp, frame_cnt
      frame_cnt = 0
    cv2.putText(annotated_image, f'{prev_frame_cnt}', org=(image_width - 30, 20), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.5, color=(100, 255, 100), lineType=2)


    annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
    cv2.imshow('Hands', annotated_image)
    if cv2.waitKey(20) & 0xFF == 27:
      # FIXME: `Listener.stop()` failed to stop mouse listener
      if DEBUG_MOUSE:
        mouse_listener.stop()
      if DEBUG_KEYBOARD:
        keyboard_listener.stop()
      break

cap.release()

# get Saver
dt = hand._dt
existence_s = hand.existence_s
handedness_s = hand.handedness_s
landmarks_s = hand.landmarks_s

## Testing

In [ ]:
np.set_printoptions(precision=4)

def dist(a, std=False, axis=1, verbose=False):
  d = np.linalg.norm(a, axis=axis)
  
  if verbose:
    print(f'mean: {np.mean(d) :.4f}, std: {np.std(d) :.4f}')

  if std:
    print(f'std: {np.std(d) :.4f}')
    d = (d - np.mean(d)) / np.std(d)

  return d

# ps = np.array(landmarks_s.x_prior[1:]).reshape(-1,21,3,2)
# xs = np.array(landmarks_s.x[1:]).reshape(-1,21,3,2)[:,:,:,0]
# dxs = np.array(landmarks_s.x[1:]).reshape(-1,21,3,2)[:,:,:,1]
xs = np.array(landmarks_s.x[1:]).reshape(-1,21,3)
dxs = np.array(landmarks_s.x[1:]).reshape(-1,21,3)
zs = np.array(landmarks_s.z[1:]).reshape(-1,21,3)
ts = (np.array(landmarks_s.t[1:]) - landmarks_s.t[0]) 
print(ts.shape)

In [ ]:
plt.figure(figsize=(30, 12))

start = 5 
end = -2

# Base Line
# plt.plot(ts[start:end], [0]*len(ts[start:end]), color='black')
# plt.plot(ts[start:end], [23]*len(ts[start:end]), color='black')
# plt.plot(ts[start:end], [30]*len(ts[start:end]), color='black')

# Distance between Tips
v = orthogonal_projection(dxs[start, 4, :] - np.mean(dxs[start, 5:17:4, :], axis=0), xs[start, 4, :] - xs[start, 6, :])[1].reshape(-1, 3)
for i in range(start, len(ts)+end-1):
  v = np.vstack([v, orthogonal_projection(dxs[i, 4, :] - np.mean(dxs[i, 5:17:4, :], axis=0), xs[i, 4, :] - xs[i, 6, :])[1].reshape(-1, 3)])
print(v.shape)
plt.plot(ts[start:end], dist(v, True), marker='.', label='d46')
plt.plot(ts[start:end], dist(xs[start:end, 4, :] - xs[start:end, 6, :], True), marker='o', label='ti_1')
# plt.plot(ts[start:end], dist(xs[start:end, 8, :, 0] - xs[start:end, 12, :, 0]), marker='.', label='im_1')
# plt.plot(ts[start:end], dist(xs[start:end, 12, :, 0] - xs[start:end, 16, :, 0]), marker='.', label='mr_1')
# plt.plot(ts[start:end], dist(xs[start:end, 16, :, 0] - xs[start:end, 20, :, 0]), marker='.', label='rp_1')
# plt.plot(ts[start:end], dist(xs[start:end, 4, :, 0] - xs[start:end, 0, :, 0]), marker='o', label='tw')
# plt.plot(ts[start:end], dist(xs[start:end, 8, :] - xs[start:end, 0, :]), marker='.', label='iw')
# plt.plot(ts[start:end], dist(xs[start:end, 12, :] - xs[start:end, 0, :]), marker='.', label='mw')
# plt.plot(ts[start:end], dist(xs[start:end, 16, :] - xs[start:end, 0, :]), marker='.', label='rw')
# plt.plot(ts[start:end], dist(xs[start:end, 20, :] - xs[start:end, 0, :]), marker='.', label='pw')

# Tips Position
# plt.plot(ts[start:end], (xs[start:end, 8, 1, 0] - np.mean(xs[start:end, 8, 1, 0])), marker='.', label='x')
# plt.plot(ts[start:end], xs[start:end, 0, 0, 0], marker='.', label='x0')
# plt.plot(ts[start:end], xs[start:end, 0, 1, 0], marker='.', label='y0')
plt.plot(ts[start:end], dist(np.mean(dxs[start:end, 5:17:4, :], axis=1)[:, 0:1], True), marker='.', label='dx')
plt.plot(ts[start:end], dist(np.mean(dxs[start:end, 5:17:4, :], axis=1)[:, 1:2], True), marker='.', label='dy')
plt.plot(ts[start:end], dist(np.mean(dxs[start:end, 5:17:4, :], axis=1)[:, 2:3], True), marker='.', label='dz')
plt.plot(ts[start:end], dist(xs[start:end, 8, 0:1], True), marker='.', label='x')
plt.plot(ts[start:end], dist(xs[start:end, 8, 1:2], True), marker='.', label='y')

# Tips Move Speed
# plt.plot(ts[start:end], xs[start:end, 8, 0, 1], marker='.', label='dx')
# plt.plot(ts[start:end], xs[start:end, 8, 1, 1], marker='.', label='dy')

# finger length
# plt.plot(ts[start:end], dist(xs[start:end, 0, :, 0] - xs[start:end, 5, :, 0]), marker='.', label='dist_05')
# plt.plot(ts[start:end], dist(xs[start:end, 0, :, 0], xs[start:end, 5, :, 0], True), marker='.', label='dist_05')

plt.legend(loc="best")

In [ ]:
from config import REAL_BONES_LENGTH

dic = {
  'raw': {},
  'norm': {},
  'factor': {},
}

factors = np.array([])
for finger_name in REAL_BONES_LENGTH.keys():
    index = REAL_BONES_LENGTH[finger_name]['index']
    length = REAL_BONES_LENGTH[finger_name]['length']
    for i in range(0, 4):
        idx_from = index[i]
        idx_to = index[i+1]
        name = f'{idx_from}_{idx_to}'
        raw = np.linalg.norm(xs[:, idx_from, :, 0] - xs[:, idx_to, :, 0], axis=1)
        factor = raw / length[i]
        dic['raw'][name] = raw
        dic['norm'][name] = raw / factor
        dic['factor'][name] = factor 
        factors = np.append(factors, factor)
factors = factors.reshape((-1, len(xs)))
avg_factor = np.mean(factors, axis=0)

In [ ]:
plt.figure(figsize=(30, 12))

start = None
end = None


for name in dic['norm'].keys():
  raw = dic['raw'][name]
  norm = dic['norm'][name]
  factor = dic['factor'][name]
plt.plot(ts[start:end], avg_factor, marker='o', color='black', label=f'avg factor')


# avg = avg[:, 1:]
# mean = np.mean(avg, axis=1)[start:end]
# std = np.std(avg, axis=1)[start:end]
# print(np.mean(mean), np.mean(std) * metrics['0_5']['base'], metrics['0_5']['base'])

# plt.plot(ts[start:end], (dist(xs[start:end, 4, :, 0] - xs[start:end, 6, :, 0], True)+1) / mean * .5, marker='o', label='ti_2')
# plt.plot(ts[start:end], (dist(xs[start:end, 8, :, 0] - xs[start:end, 12, :, 0], True)+1) / mean * .5, marker='o', label='im_2')
# plt.plot(ts[start:end], (dist(xs[start:end, 12, :, 0] - xs[start:end, 16, :, 0], True)+1) / mean * .5, marker='o', label='mr_2')
# plt.plot(ts[start:end], (dist(xs[start:end, 16, :, 0] - xs[start:end, 20, :, 0], True)+1) / mean * .5, marker='o', label='rp_2')

# mean = mean * metrics['0_5']['base']
# std *= metrics['0_5']['base']

# plt.plot(ts[start:end], mean, marker='o', color='black', label='avg')
# plt.fill_between(ts[start:end], mean - std, mean + std, color='gray', alpha=.5)

plt.legend(loc='best')

In [ ]:
zs = np.asanyarray(landmarks_s.z[1:]).reshape(-1, 21, 3)#[:-1]
prior = np.asanyarray(landmarks_s.x_prior[1:]).reshape(-1, 21, 3, 2)#[1:]
post = np.asanyarray(landmarks_s.x_post[1:]).reshape(-1, 21, 3, 2)
ts = np.arange(zs.shape[0]) * dt
print(post.shape, ts.shape)

In [ ]:
plt.figure(figsize=(20, 15))
# yv: 100
idx = 8
xyz = 1
start = 0
end = -1

plt.subplot(611)
plt.plot(ts[start:end], zs[start:end, idx, 0], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 0, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 0, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('x pos')

plt.subplot(612)
plt.plot(ts[start:end], prior[start:end, idx, 0, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 0, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('x vel')

plt.subplot(613)
plt.plot(ts[start:end], zs[start:end, idx, 1], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 1, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 1, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('y pos')

plt.subplot(614)
plt.plot(ts[start:end], prior[start:end, idx, 1, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 1, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('y vel')

plt.subplot(615)
plt.plot(ts[start:end], zs[start:end, idx, 2], marker='.', label='z')
plt.plot(ts[start:end], prior[start:end, idx, 2, 0], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 2, 0], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('z pos')

plt.subplot(616)
plt.plot(ts[start:end], prior[start:end, idx, 2, 1], marker='.', label='prior')
plt.plot(ts[start:end], post[start:end, idx, 2, 1], marker='.', label='post')

plt.xticks(np.arange(zs.shape[0]*dt, step=6*dt))
plt.legend(loc='best')
plt.title('z vel')

In [ ]:
start = None
end = None
# start = int(1.1/dt)
# end = int(10/dt)

zs_std = np.std(zs[start:end], axis=0)
zs_mean = np.mean(zs[start:end], axis=0)
prior_std = np.std(prior[start:end], axis=0)
prior_mean = np.mean(prior[start:end], axis=0)
post_std = np.std(post, axis=0)[start:end]
post_mean = np.mean(post[start:end], axis=0)
print(zs_std.shape, prior_std.shape, post_std.shape)

In [ ]:
plt.figure(figsize=(20, 6))

idx = 8

# mean & std - pos
# plt.errorbar(np.arange(21)+.0, np.zeros(21), yerr=zs_std[:, 0], linestyle='None', fmt='o', label='z-x')
# plt.errorbar(np.arange(21)+.1, np.zeros(21), yerr=zs_std[:, 1], linestyle='None', fmt='o', label='z-y')
# plt.errorbar(np.arange(21)+.2, np.zeros(21), yerr=zs_std[:, 2], linestyle='None', fmt='o', label='z-z')
# plt.errorbar(np.arange(21)+.3, np.zeros(21), yerr=prior_std[:, 0, 0], linestyle='None', fmt='o', label='prior-x')
# plt.errorbar(np.arange(21)+.4, np.zeros(21), yerr=prior_std[:, 1, 0], linestyle='None', fmt='o', label='prior-y')
# plt.errorbar(np.arange(21)+.5, np.zeros(21), yerr=prior_std[:, 2, 0], linestyle='None', fmt='o', label='prior-z')
# plt.errorbar(np.arange(21)+.6, np.zeros(21), yerr=post_std[:, 0, 0], linestyle='None', fmt='o', label='post-x')
# plt.errorbar(np.arange(21)+.7, np.zeros(21), yerr=post_std[:, 1, 0], linestyle='None', fmt='o', label='post-y')
# plt.errorbar(np.arange(21)+.8, np.zeros(21), yerr=post_std[:, 2, 0], linestyle='None', fmt='o', label='post-z')
# mean & std - vel
# plt.errorbar(np.arange(21)+.3, np.zeros(21), yerr=prior_std[:, 0, 1], linestyle='None', fmt='o', label='prior-x')
# plt.errorbar(np.arange(21)+.4, np.zeros(21), yerr=prior_std[:, 1, 1], linestyle='None', fmt='o', label='prior-y')
# plt.errorbar(np.arange(21)+.5, np.zeros(21), yerr=prior_std[:, 2, 1], linestyle='None', fmt='o', label='prior-z')
# plt.errorbar(np.arange(21)+.6, np.zeros(21), yerr=post_std[:, 0, 1], linestyle='None', fmt='o', label='post-x')
# plt.errorbar(np.arange(21)+.7, np.zeros(21), yerr=post_std[:, 1, 1], linestyle='None', fmt='o', label='post-y')
# plt.errorbar(np.arange(21)+.8, np.zeros(21), yerr=post_std[:, 2, 1], linestyle='None', fmt='o', label='post-z')

# pos
print((prior[start:end, idx, :, 0] - np.mean(post[start:end, idx, :, 0], axis=0)).shape)
print(np.std((prior[start:end, idx, :, 0] - np.mean(post[start:end, idx, :, 0], axis=0)), axis=0))
# print(np.mean(post[start:end, idx, :, 0], axis=0))
plt.plot(ts[start:end], (prior[start:end, idx, :, 0] - np.mean(post[start:end, idx, :, 0], axis=0))[:, 0], marker='.', label=f'pos-prior-{idx}')
# plt.plot(ts[start:end], prior[start:end, idx, 0, 0], marker='.', label=f'pos-prior-{idx}')
# plt.plot(ts[start:end], zs[start:end, idx, 0], marker='.', label=f'pos-z-{idx}')
# plt.plot(ts[start:end], post[start:end, idx, 0, 0], marker='.', label=f'pos-post-{idx}')

# vel 
# plt.plot(ts[start:end], prior[start:end, idx, 0, 1], marker='.', label=f'vel-prior-{idx}')
# plt.plot(ts[start:end], ((zs - np.roll(zs, 1, axis=0))/dt)[start:end, idx, 0], marker='.', label=f'vel-z-{idx}')

# plt.xticks(np.arange(21))
plt.legend()
print(np.mean(prior_std[:, :, 1], axis=0))

In [ ]:
plt.figure(figsize=(12,6))
np.set_printoptions(suppress=True)
print(np.round(prior_std[:, :, 1], 3))
plt.plot(np.arange(21), prior_std[:, 0, 1], marker='.', label='x')
plt.plot(np.arange(21), prior_std[:, 1, 1], marker='.', label='y')
plt.plot(np.arange(21), prior_std[:, 2, 1], marker='.', label='z')
plt.xticks(np.arange(21))
plt.legend()
plt.show()

In [ ]:
max_v = np.max(prior_std[:,:,1], axis=0)

# print(max_v)
ratio = np.round(prior_std[:,:,1]/max_v, 4)
print(ratio)

plt.figure(figsize=(10,6))
plt.plot(np.arange(21), ratio[:, 0], marker='.', label='x')
plt.plot(np.arange(21), ratio[:, 1], marker='.', label='y')
plt.plot(np.arange(21), ratio[:, 2], marker='.', label='z')
plt.xticks(np.arange(21))
plt.legend()
plt.show()